In [ ]:
import os
import pandas as pd
import psycopg2
from ydata_profiling import ProfileReport

# Папка для сохранения отчетов
output_dir = "/Users/gulimoh/collateral-assessment/stat/reports"
os.makedirs(output_dir, exist_ok=True)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="5837",
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос
query = """
SELECT id, number_of_rooms, total_area, price, currency, regionname, cityname, createdtime
FROM apartments
WHERE regionname = 'Ташкентская область'
    AND createdtime >= '2024-02-01'
    AND total_area >= 18;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()

# Курс валют
usd_to_uzs = 12600  # Актуальный курс

# Функция конвертации цен
def convert_prices(row):
    if row["currency"] == "UYE":
        price_uye = row["price"]
        price_uz = price_uye * usd_to_uzs
    else:
        price_uz = row["price"]
        price_uye = price_uz / usd_to_uzs

    # Проверяем, является ли цена за м²
    if price_uye <= 5000:
        price_per_m2_uye = price_uye
        price_uye *= row["total_area"]
    else:
        price_per_m2_uye = price_uye / row["total_area"]

    if price_uz <= 65003698:
        price_per_m2_uz = price_uz
        price_uz *= row["total_area"]
    else:
        price_per_m2_uz = price_uz / row["total_area"]

    return pd.Series([price_uye, price_uz, price_per_m2_uye, price_per_m2_uz])

# Применяем конвертацию
df[["price_uye", "price_uz", "price_per_m2_uye", "price_per_m2"]] = df.apply(convert_prices, axis=1)

# Удаляем колонку "currency"
df.drop(columns=["currency"], inplace=True)

# Фильтрация по total_area (1-4 комнаты: 18-150, 5+: 18-400)
df = df[((df["number_of_rooms"] <= 4) & (df["total_area"] <= 150)) | ((df["number_of_rooms"] >= 5) & (df["total_area"] <= 400))]

# Минимальные цены для разных типов квартир
min_price_uye = {1: 5000, 2: 8000, 3: 10000, 4: 12000, "5+": 15000}

# Разделение на Ташкент и Ташобласть
df_tashkent = df[df["cityname"] == "Ташкент"].copy()
df_tashoblast = df[df["cityname"] != "Ташкент"].copy()

# ✅ Анализ квартир ≤ 5000 UYE и < 40 м² (до фильтрации min_price_uye)
df_small_cheap_tashkent = df_tashkent[(df_tashkent["price_uye"] <= 5000) & (df_tashkent["total_area"] < 40)].copy()
df_small_cheap_tashoblast = df_tashoblast[(df_tashoblast["price_uye"] <= 5000) & (df_tashoblast["total_area"] < 40)].copy()

df_small_expensive_tashkent = df_tashkent[(df_tashkent["price_uye"] > 5000) & (df_tashkent["total_area"] < 40)].copy()
df_small_expensive_tashoblast = df_tashoblast[(df_tashoblast["price_uye"] > 5000) & (df_tashoblast["total_area"] < 40)].copy()

# Фильтрация по минимальной цене
for rooms, min_price in min_price_uye.items():
    if rooms == "5+":
        df_tashkent = df_tashkent[~((df_tashkent["number_of_rooms"] >= 5) & (df_tashkent["price_uye"] < min_price))]
        df_tashoblast = df_tashoblast[~((df_tashoblast["number_of_rooms"] >= 5) & (df_tashoblast["price_uye"] < min_price))]
    else:
        df_tashkent = df_tashkent[~((df_tashkent["number_of_rooms"] == rooms) & (df_tashkent["price_uye"] < min_price))]
        df_tashoblast = df_tashoblast[~((df_tashoblast["number_of_rooms"] == rooms) & (df_tashoblast["price_uye"] < min_price))]

# Функция для удаления аутлайеров
def remove_outliers(df):
    if df.empty:
        return df  # Возвращаем пустой df, если данных нет

    lower_bound = df["price_uye"].quantile(0.03)
    upper_bound = df["price_uye"].quantile(0.96)

    return df[(df["price_uye"] > lower_bound) & (df["price_uye"] < upper_bound)]

# Убираем аутлайеры
df_tashkent_clean = remove_outliers(df_tashkent)
df_tashoblast_clean = remove_outliers(df_tashoblast)

# Функция для сохранения отчетов
def generate_report(df, name):
    if not df.empty:
        profile = ProfileReport(df, title=f"Profiling Report: {name}", explorative=True)
        filename = os.path.join(output_dir, f"{name}.html")
        profile.to_file(filename)
        print(f"✅ {filename} saved")

# Создаем отчеты по комнатам
for city, df_city in [("Ташкент", df_tashkent_clean), ("Ташобласть", df_tashoblast_clean)]:
    for rooms in [1, 2, 3, 4, "5+"]:
        if rooms == "5+":
            df_filtered = df_city[df_city["number_of_rooms"] >= 5]
        else:
            df_filtered = df_city[df_city["number_of_rooms"] == rooms]

        generate_report(df_filtered, f"{city}_{rooms}_комнат")

# Анализ аутлайеров (верхних и нижних)
for city, df_city_clean, df_city_raw in [
    ("Ташкент", df_tashkent_clean, df_tashkent),
    ("Ташобласть", df_tashoblast_clean, df_tashoblast)
]:
    if not df_city_raw.empty:
        lower_bound = df_city_raw["price_uye"].quantile(0.03)
        upper_bound = df_city_raw["price_uye"].quantile(0.96)

        df_upper_outliers = df_city_raw[df_city_raw["price_uye"] >= upper_bound]
        df_lower_outliers = df_city_raw[df_city_raw["price_uye"] <= lower_bound]

        generate_report(df_upper_outliers, f"{city}_Верхние_аутлайеры")
        generate_report(df_lower_outliers, f"{city}_Нижние_аутлайеры")

        # Разделение аутлайеров по комнатам
        for rooms in [1, 2, 3, 4, "5+"]:
            if rooms == "5+":
                df_upper = df_upper_outliers[df_upper_outliers["number_of_rooms"] >= 5]
                df_lower = df_lower_outliers[df_lower_outliers["number_of_rooms"] >= 5]
            else:
                df_upper = df_upper_outliers[df_upper_outliers["number_of_rooms"] == rooms]
                df_lower = df_lower_outliers[df_lower_outliers["number_of_rooms"] == rooms]

            generate_report(df_upper, f"{city}_Верхние_аутлайеры_{rooms}_комнат")
            generate_report(df_lower, f"{city}_Нижние_аутлайеры_{rooms}_комнат")

# Проверяем количество записей перед созданием отчетов
print(f"Маленькие и дешевые квартиры в Ташкенте: {df_small_cheap_tashkent.shape[0]} записей")
print(f"Маленькие и дешевые квартиры в Ташобласти: {df_small_cheap_tashoblast.shape[0]} записей")
print(f"Маленькие и дорогие квартиры в Ташкенте: {df_small_expensive_tashkent.shape[0]} записей")
print(f"Маленькие и дорогие квартиры в Ташобласти: {df_small_expensive_tashoblast.shape[0]} записей")

# Генерация отчетов только если есть данные
if not df_small_cheap_tashkent.empty:
    generate_report(df_small_cheap_tashkent, "Ташкент_Маленькие_и_дешевые")
if not df_small_cheap_tashoblast.empty:
    generate_report(df_small_cheap_tashoblast, "Ташобласть_Маленькие_и_дешевые")
if not df_small_expensive_tashkent.empty:
    generate_report(df_small_expensive_tashkent, "Ташкент_Маленькие_и_дорогие")
if not df_small_expensive_tashoblast.empty:
    generate_report(df_small_expensive_tashoblast, "Ташобласть_Маленькие_и_дорогие")

print("🎯 Все отчеты сохранены в:", output_dir)

### Общий отчет квартиры в  Бухаре, Самарканде, Навоий и  Фергане without outlier


In [3]:
import pandas as pd
import psycopg2
from ydata_profiling import ProfileReport

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Заменить на актуальный пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос
query = """
SELECT number_of_rooms, total_area, price, currency, regionname, cityname, createdtime, isactive,
       floor, total_floors, furnished, repairs, comission, wc, house_type
FROM apartments
WHERE total_area IS NOT NULL AND total_area >= 18 AND total_area <= 400
    AND price IS NOT NULL AND price > 0
    AND createdtime >= '2024-02-01';
"""

# Загружаем данные в DataFrame
df = pd.read_sql(query, conn)
conn.close()

# Курс валют
usd_to_uzs = 12600  # Актуальный курс

def convert_prices(row):
    if row["currency"] == "UYE":
        price_uye = row["price"]
        price_uz = price_uye * usd_to_uzs
    else:
        price_uz = row["price"]
        price_uye = price_uz / usd_to_uzs

    if price_uye <= 5000:
        price_per_m2_uye = price_uye
        price_uye *= row["total_area"]
    else:
        price_per_m2_uye = price_uye / row["total_area"]

    if price_uz <= 65003698:
        price_per_m2_uz = price_uz
        price_uz *= row["total_area"]
    else:
        price_per_m2_uz = price_uz / row["total_area"]

    return pd.Series([price_uye, price_uz, price_per_m2_uye, price_per_m2_uz])

# Применяем конвертацию цен
df[["price_uye", "price_uz", "price_per_m2_uye", "price_per_m2_uz"]] = df.apply(convert_prices, axis=1)
df.drop(columns=["currency", "price"], inplace=True)

# Фильтрация по площади
filters = {
    1: (18, 70),
    2: (25, 100),
    3: (30, 150),
    4: (50, 150),
    5: (80, 400)
}

def apply_filters(df):
    def get_min_area(rooms):
        return filters.get(rooms, (0, float("inf")))[0] if rooms < 5 else filters[5][0]

    def get_max_area(rooms):
        return filters.get(rooms, (0, float("inf")))[1] if rooms < 5 else filters[5][1]

    return df[
        (df["total_area"] >= df["number_of_rooms"].map(get_min_area)) &
        (df["total_area"] <= df["number_of_rooms"].map(get_max_area))
        ]

# Применяем фильтрацию по площади
df = apply_filters(df)

# Фильтруем нужные регионы
regions_of_interest = ["Бухарская область", "Самаркандская область", "Навоийская область", "Ферганская область"]
df_filtered = df[df["regionname"].isin(regions_of_interest)]

# Функция для генерации отчета
def generate_report(data, filename):
    report = ProfileReport(data, explorative=True)
    report.to_file(filename)

# Генерируем общий отчет по каждому региону
for region in df_filtered["regionname"].unique():
    df_region = df_filtered[df_filtered["regionname"] == region]
    generate_report(df_region, f"full_report_{region}.html")

print("Общие отчеты по регионам созданы.")


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_36110/820305082.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 267.03it/s]
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See t

Общие отчеты по регионам созданы.


### Квартиры в  Бухаре, Самарканде, Навоий и  Фергане with outliers


In [32]:
import pandas as pd
import psycopg2
from ydata_profiling import ProfileReport

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Заменить на актуальный пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос
query = """
SELECT number_of_rooms, total_area, price, currency, regionname, cityname, createdtime, isactive, floor, total_floors, furnished, repairs, comission, wc, house_type
FROM apartments
WHERE total_area IS NOT NULL AND total_area >= 18 AND total_area <= 400
    AND price IS NOT NULL AND price > 0
    AND createdtime >= '2024-02-01';
"""

# Загружаем данные в DataFrame
df = pd.read_sql(query, conn)
conn.close()

# Ограничение значений этажей до 25
df["floor"] = df["floor"].clip(upper=25)
df["total_floors"] = df["total_floors"].clip(upper=25)

# Курс валют
usd_to_uzs = 12600  # Актуальный курс

def convert_prices(row):
    if row["currency"] == "UYE":
        price_uye = row["price"]
        price_uz = price_uye * usd_to_uzs
    else:
        price_uz = row["price"]
        price_uye = price_uz / usd_to_uzs

    if price_uye <= 5000:
        price_per_m2_uye = price_uye
        price_uye *= row["total_area"]
    else:
        price_per_m2_uye = price_uye / row["total_area"]

    if price_uz <= 65003698:
        price_per_m2_uz = price_uz
        price_uz *= row["total_area"]
    else:
        price_per_m2_uz = price_uz / row["total_area"]

    return pd.Series([price_uye, price_uz, price_per_m2_uye, price_per_m2_uz])

# Применяем конвертацию к DataFrame
df[["price_uye", "price_uz", "price_per_m2_uye", "price_per_m2_uz"]] = df.apply(convert_prices, axis=1)

df.drop(columns=["currency", "price"], inplace=True)


# Фильтрация по площади
filters = {
    1: (18, 70),
    2: (25, 100),
    3: (30, 150),
    4: (50, 150),
    5: (80, 400)
}

def apply_filters(df):
    def get_min_area(rooms):
        return filters.get(rooms, (0, float("inf")))[0] if rooms < 5 else filters[5][0]

    def get_max_area(rooms):
        return filters.get(rooms, (0, float("inf")))[1] if rooms < 5 else filters[5][1]

    return df[
        (df["total_area"] >= df["number_of_rooms"].map(get_min_area)) &
        (df["total_area"] <= df["number_of_rooms"].map(get_max_area))
        ]


# Применяем фильтрацию по площади перед удалением выбросов
df = apply_filters(df)

# Удаление выбросов
outlier_bounds = {
    "Бухарская область": (150, 1200),
    "Самаркандская область": (300, 5000),
    "Навоийская область": (200, 1200),
    "Ферганская область": (200, 1200)
}

def identify_outliers(row):
    if row["regionname"] in outlier_bounds:
        low, high = outlier_bounds[row["regionname"]]
        return row["price_per_m2_uye"] < low or row["price_per_m2_uye"] > high
    return False

df["is_outlier"] = df.apply(identify_outliers, axis=1)

# Сохраняем копию DataFrame до удаления выбросов
df_before_outliers = df.copy()

# Удаляем выбросы
df = df[~df["is_outlier"]]

# Разделяем выбросы и нормальные данные
df_outliers = df_before_outliers[df_before_outliers["is_outlier"]]

# Фильтруем только нужные регионы
regions_of_interest = ["Бухарская область", "Самаркандская область", "Навоийская область", "Ферганская область"]
df = df[df["regionname"].isin(regions_of_interest)]
df_outliers = df_outliers[df_outliers["regionname"].isin(regions_of_interest)]

def generate_report(data, filename):
    report = ProfileReport(data, explorative=True)
    report.to_file(filename)

# Генерация отчетов до и после удаления выбросов
for region in df["regionname"].unique():
    for rooms in [1, 2, 3, 4, '5+']:
        if rooms == '5+':
            df_subset_before = df_before_outliers[(df_before_outliers["regionname"] == region) & (df_before_outliers["number_of_rooms"] >= 5)]
            df_subset_after = df[(df["regionname"] == region) & (df["number_of_rooms"] >= 5)]
        else:
            df_subset_before = df_before_outliers[(df_before_outliers["regionname"] == region) & (df_before_outliers["number_of_rooms"] == rooms)]
            df_subset_after = df[(df["regionname"] == region) & (df["number_of_rooms"] == rooms)]

        if not df_subset_before.empty:
            generate_report(df_subset_before, f"before_outliers_{region}_{rooms}rooms.html")
        if not df_subset_after.empty:
            generate_report(df_subset_after, f"after_outliers_{region}_{rooms}rooms.html")

# Генерация отчетов для выбросов по областям
for region in df_outliers["regionname"].unique():
    low, high = outlier_bounds[region]

    # Низкие выбросы
    df_low_outliers = df_outliers[(df_outliers["regionname"] == region) & (df_outliers["price_per_m2_uye"] < low)]
    if not df_low_outliers.empty:
        generate_report(df_low_outliers, f"outliers_{region}_price_per_m2_uye_low.html")

    # Высокие выбросы
    df_high_outliers = df_outliers[(df_outliers["regionname"] == region) & (df_outliers["price_per_m2_uye"] > high)]
    if not df_high_outliers.empty:
        generate_report(df_high_outliers, f"outliers_{region}_price_per_m2_uye_high.html")

print("Отчеты успешно созданы.")


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_36110/4151163792.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 313.76it/s]
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See 

Отчеты успешно созданы.


### Квартиры в Ташкенте и Ташобласть with outliers

In [7]:
import os
import pandas as pd
import psycopg2
from ydata_profiling import ProfileReport

# Папка для сохранения отчетов
output_dir = "/Users/gulimoh/collateral-assessment/stat/chek"
os.makedirs(output_dir, exist_ok=True)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Укажи свой пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос
query = """
SELECT number_of_rooms, total_area, price, currency, regionname, cityname, createdtime,
       isactive, floor, total_floors, furnished, repairs, comission, wc, house_type
FROM apartments
WHERE regionname = 'Ташкентская область'
    AND createdtime >= '2024-02-01'
    AND total_area >= 18;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()


# Курс валют
usd_to_uzs = 12600  # Актуальный курс

def convert_prices(row):
    if row["currency"] == "UYE":
        price_uye = row["price"]
        price_uz = price_uye * usd_to_uzs
    else:
        price_uz = row["price"]
        price_uye = price_uz / usd_to_uzs

    if price_uye <= 5000:
        price_per_m2_uye = price_uye
        price_uye *= row["total_area"]
    else:
        price_per_m2_uye = price_uye / row["total_area"]

    if price_uz <= 65003698:
        price_per_m2_uz = price_uz
        price_uz *= row["total_area"]
    else:
        price_per_m2_uz = price_uz / row["total_area"]

    return pd.Series([price_uye, price_uz, price_per_m2_uye, price_per_m2_uz])

# Применяем конвертацию
df[["price_uye", "price_uz", "price_per_m2_uye", "price_per_m2_uz"]] = df.apply(convert_prices, axis=1)

# Удаляем колонку "currency" и "price"
df.drop(columns=["currency", "price"], inplace=True)

# Разделение на Ташкент и Ташобласть
df_tashkent = df[df["cityname"] == "Ташкент"].copy()
df_tashoblast = df[df["cityname"] != "Ташкент"].copy()

# Ограничение этажности в Ташобласти
df_tashoblast["floor"] = df_tashoblast["floor"].clip(upper=25)
df_tashoblast["total_floors"] = df_tashoblast["total_floors"].clip(upper=25)

# Фильтрация по total_area
filters = {
    1: (18, 70),
    2: (25, 100),
    3: (30, 150),
    4: (50, 150),
    5: (80, 400)
}

def apply_filters(df):
    def get_min_area(rooms):
        return filters.get(rooms, (0, float("inf")))[0] if rooms < 5 else filters[5][0]

    def get_max_area(rooms):
        return filters.get(rooms, (0, float("inf")))[1] if rooms < 5 else filters[5][1]

    return df[
        (df["total_area"] >= df["number_of_rooms"].map(get_min_area)) &
        (df["total_area"] <= df["number_of_rooms"].map(get_max_area))
        ]

df_tashkent = apply_filters(df_tashkent)
df_tashoblast = apply_filters(df_tashoblast)


# Удаление выбросов
outlier_bounds = {
    "tashkent": (700, 5000),
    "tashoblast": (300, 3000)
}

def remove_outliers(df, min_val, max_val):
    return df[(df['price_per_m2_uye'] >= min_val) & (df['price_per_m2_uye'] <= max_val)]

df_tashkent_clean = remove_outliers(df_tashkent, *outlier_bounds["tashkent"])
df_tashoblast_clean = remove_outliers(df_tashoblast, *outlier_bounds["tashoblast"])

# Сохранение выбросов
outliers_tashkent = df_tashkent[~df_tashkent.index.isin(df_tashkent_clean.index)]
outliers_tashoblast = df_tashoblast[~df_tashoblast.index.isin(df_tashoblast_clean.index)]

# Генерация отчетов по выбросам (нижние и верхние)
for df_name, df_outliers, bounds in [
    ("tashkent", outliers_tashkent, outlier_bounds["tashkent"]),
    ("tashoblast", outliers_tashoblast, outlier_bounds["tashoblast"])
]:
    min_bound, max_bound = bounds
    room_categories = [1, 2, 3, 4, "5plus"] if df_name == "tashkent" else [1, 2, 3, "4plus"]

    for rooms in room_categories:
        if rooms == "5plus":
            df_subset = df_outliers[df_outliers["number_of_rooms"] >= 5]
        elif rooms == "4plus":
            df_subset = df_outliers[df_outliers["number_of_rooms"] >= 4]
        else:
            df_subset = df_outliers[df_outliers["number_of_rooms"] == rooms]

        df_lower = df_subset[df_subset["price_per_m2_uye"] < min_bound]
        df_upper = df_subset[df_subset["price_per_m2_uye"] > max_bound]

        if not df_lower.empty:
            profile_lower = ProfileReport(df_lower, minimal=True)
            profile_lower.to_file(f"{output_dir}/outliers_{df_name}_{rooms}_price_per_m2_uye_{min_bound}.html")

        if not df_upper.empty:
            profile_upper = ProfileReport(df_upper, minimal=True)
            profile_upper.to_file(f"{output_dir}/outliers_{df_name}_{rooms}_price_per_m2_uye_{max_bound}.html")

# Генерация отчетов по основным данным
for df_name, df_clean in [
    ("tashkent", df_tashkent_clean),
    ("tashoblast", df_tashoblast_clean)
]:
    room_categories = [1, 2, 3, 4, "5plus"] if df_name == "tashkent" else [1, 2, 3, "4plus"]

    for rooms in room_categories:
        df_subset = df_clean[df_clean["number_of_rooms"] == rooms] if isinstance(rooms, int) else   df_clean[df_clean["number_of_rooms"] >= int(rooms.replace("plus", ""))]
        profile = ProfileReport(df_subset, minimal=True)
        profile.to_file(f"{output_dir}/{df_name}_{rooms}rooms.html")

/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_36110/2047310121.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 866.41it/s]
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See 